## Data Loader
[인공지능 기술·산업 생태계 육성방안
연구](https://spri.kr/posts/view/23669)



In [17]:
from settings import get_settings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

settings = get_settings()

def create_model()-> ChatOpenAI:
    return ChatOpenAI(
        model=settings.default_model,
        base_url=settings.base_url,
        api_key=settings.api_key,
    )


def create_embeddings() -> OpenAIEmbeddings:
    return OpenAIEmbeddings(
        model=settings.embedding_model,
        base_url=settings.base_url,
        api_key=settings.api_key,
        check_embedding_ctx_length=False
    )

llm = create_model()
embeddings = create_embeddings()


설정

In [18]:
import pdfplumber
from pathlib import Path

FILE_PATH = Path.cwd().parent/"data"/"RE-185. 인공지능 기술·산업 생태계 육성방안 연구.pdf"

text = ""

with pdfplumber.open(FILE_PATH) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

In [19]:
text[:50]

'연구보고서 RE-185\n인공지능 기술·산업 생태계 육성방안\n연구\nA Study on the'

In [20]:
from utils import chunk_text

chunks = chunk_text(text, 500, 40)
print(len(chunks))
print(chunks[0])


290
연구보고서 RE-185
인공지능 기술·산업 생태계 육성방안
연구
A Study on the Promotion Policy for the Korean Technological and
Industirial Ecosystem in Artificial Intelligence
봉강호 / 안성원
2025. 4.이 보고서는 2024년도 과학기술정보통신부 정보통신진흥기금을 지원
받아 수행한 연구결과로 보고서 내용은 연구자의 견해이며, 과학기술정보
통신부의 공식입장과 다를 수 있습니다.목 차
제1장 서론 ·························································································································· 1
제1절 연구 배경 및 필요성 ··························································································


In [21]:
print(len(chunks))

290


In [22]:
vectors = embeddings.embed_documents(chunks)

print(vectors[0][:3])
print(len(vectors))
print(len(vectors[0]))

[-0.05807943269610405, 0.011216574348509312, -0.007120043970644474]
290
1024


Neo4j Driver 연결한다.

In [27]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(settings.neo4j_url, auth=(settings.neo4j_user, settings.neo4j_password))

try:
    driver.verify_connectivity()
    print("Neo4j 연결 성공")
except Exception as e:
    print(f"연결 실패: {e}")

Neo4j 연결 성공


Neo4J Index를 생성한다.
```cypher
CREATE VECTOR INDEX pdf
IF NOT EXISTS FOR (c:Chunk) ON c.embedding
```

In [30]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CYPHER_DIR = PROJECT_ROOT / "cypher"

query_path = CYPHER_DIR / "001-create-vector-index.cypher"


In [ ]:
query_path = (
    Path.cwd()
    / "cypher"
    / "001-create-vector-index.cypher"
)

query = query_path.read_text(encoding="utf-8")
driver.execute_query(query, database_="neo4j")


In [34]:
query_path = (
    Path.cwd()
    / "cypher"
    / "002-show-index.cypher"
)

query = query_path.read_text(encoding="utf-8")
records, summary, keys = driver.execute_query(query, database_="neo4j")

In [36]:
records

[<Record name='pdf' state='ONLINE' labelsOrTypes=['Chunk'] properties=['embedding'] type='VECTOR'>]

In [37]:
summary

In [38]:

for record in records:
    print(record.data())

{'name': 'pdf', 'state': 'ONLINE', 'labelsOrTypes': ['Chunk'], 'properties': ['embedding'], 'type': 'VECTOR'}
